In [2]:
import pandas as pd 

In [3]:
df=pd.read_csv("jsb_chorales/train/chorale_000.csv")
df

,note0,note1,note2,note3
0,74,70,65,58
1,74,70,65,58
2,74,70,65,58
3,74,70,65,58
4,75,70,58,55
...,...,...,...,...
187,70,65,62,46
188,70,65,62,46
189,70,65,62,46
190,70,65,62,46


In [4]:
import os 
train_file=sorted([os.path.join('jsb_chorales','train',f) for f in os.listdir(os.path.join('jsb_chorales','train')) if f.endswith('.csv') ])
test_file=sorted([os.path.join('jsb_chorales','test',f) for f in os.listdir(os.path.join('jsb_chorales','test')) if f.endswith('.csv') ])
valid_file=sorted([os.path.join('jsb_chorales','valid',f) for f in os.listdir(os.path.join('jsb_chorales','valid')) if f.endswith('.csv') ])

In [5]:
train_data=[pd.read_csv(f).values.tolist() for f in train_file]
test_data=[pd.read_csv(f).values.tolist() for f in test_file]
valid_data=[pd.read_csv(f).values.tolist() for f in valid_file]

In [6]:
from music21 import stream,chord

chorale=train_data[0]
s=stream.Stream()
for row in chorale:
    s.append(chord.Chord([n for n in row if n],quarterLength=1))

s.show('midi')

In [7]:
import numpy as np 
min_note ,max_note=36,81 #chorales minimum and maximum value
window_size,window_offset,batch_size=32,16,32

def make_xy(chorales):
    windows=[c[i:i + window_size+1] for c in chorales for i in range(0,len(c)-window_size,window_offset)]
    data=np.array(windows,dtype=int)
    data=np.where(data==0,0,data-min_note+1)
    data=np.clip(data,0,max_note-min_note+1)
    flat=data.reshape(data.shape[0],-1)
    return flat[:,:-1],flat[:,1:]

X_train,y_train=make_xy(train_data)
X_test,y_test=make_xy(test_data)
X_valid,y_valid=make_xy(valid_data)


In [8]:
y_train.shape

(3111, 131)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D,Dense,LSTM,Dropout,Embedding,BatchNormalization
from tensorflow.keras.optimizers import Nadam

In [10]:
model=Sequential()

model.add(Embedding(input_dim=47,output_dim=5,input_shape=[None]))
model.add(Conv1D(32,kernel_size=2,padding='causal',activation='relu'))
model.add(BatchNormalization())
model.add(Conv1D(48,kernel_size=2,padding='causal',activation='relu',dilation_rate=2))
model.add(BatchNormalization())
model.add(Conv1D(64,kernel_size=2,padding='causal',activation='relu',dilation_rate=4))
model.add(BatchNormalization())
model.add(Conv1D(96,kernel_size=2,padding='causal',activation='relu',dilation_rate=8))
model.add(BatchNormalization())
model.add(Conv1D(128,kernel_size=2,padding='causal',activation='relu',dilation_rate=16))
model.add(BatchNormalization())
model.add(Dropout(0.05))
model.add(LSTM(256,return_sequences=True))
model.add(Dense(47,activation='softmax'))

model.summary()

/Users/anushkapanwar/Desktop/Jenv/jvenv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:126: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 5)        │           235 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, None, 32)       │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, None, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, None, 48)       │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, None, 48)       │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, None, 64)       │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, None, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, None, 96)       │        12,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, None, 96)       │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, None, 128)      │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, None, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, None, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 256)      │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 47)       │        12,079 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 454,794 (1.73 MB)

 Trainable params: 454,058 (1.73 MB)

 Non-trainable params: 736 (2.88 KB)

In [12]:
optimizer = Nadam(learning_rate=1e-3)
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
              metrics=["accuracy"])
model.fit(X_train,y_train, epochs=20, validation_data=[X_valid,y_valid],batch_size=batch_size)

Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 20s 174ms/step - accuracy: 0.5674 - loss: 1.6664 - val_accuracy: 0.0320 - val_loss: 3.8667
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 16s 160ms/step - accuracy: 0.7754 - loss: 0.8390 - val_accuracy: 0.0356 - val_loss: 4.1220
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 16s 162ms/step - accuracy: 0.8017 - loss: 0.7033 - val_accuracy: 0.0989 - val_loss: 3.7246
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 16s 166ms/step - accuracy: 0.8165 - loss: 0.6310 - val_accuracy: 0.1919 - val_loss: 3.2571
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 17s 178ms/step - accuracy: 0.8283 - loss: 0.5812 - val_accuracy: 0.3331 - val_loss: 2.3858
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 18s 185ms/step - accuracy: 0.8369 - loss: 0.5445 - val_accuracy: 0.7554 - val_loss: 0.8597
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 19s 190ms/step - accuracy: 0.8458 - loss: 0.5084 - val_accuracy: 0.8056 - val_loss: 0.6741
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 18s 188ms/step - accuracy: 0.8538 - loss: 0.4786 - val_accu

In [13]:
def sample_next_note(probs):
    probability=np.asarray(probs,dtype=float)
    prob_sum=probability.sum()
    if prob_sum <=0 or not np.isfinite(prob_sum):
        return int(np.argmax(probability))
    probability/=prob_sum
    return np.random.choice(len(probability),p=probability)

In [14]:
def generate_chorale(model,seed_chords,length):
    token_sequence=np.array(seed_chords,dtype=int)
    token_sequence=np.where(token_sequence==0,0,token_sequence-min_note+1)
    token_sequence=token_sequence.reshape(1,-1)

    for _ in range(length * 4):
        next_token_probabilities=model.predict(token_sequence)[0,-1]
        next_token=sample_next_note(next_token_probabilities)
        token_sequence=np.concatenate([token_sequence,[[next_token]]],axis=1)
    
    token_sequence=np.where(token_sequence==0,0,token_sequence+min_note-1)

    return token_sequence.reshape(-1,4)



In [26]:
seed_chord=test_data[12][:8]

chorale =seed_chord
s=stream.Stream()
for row in chorale:
    s.append(chord.Chord([n for n in row if n],quarterLength=1))
s.show('midi')

In [27]:
new_chorale=generate_chorale(model,seed_chord,56)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━

In [28]:
seed_chord=new_chorale.tolist()

chorale =seed_chord
s=stream.Stream()
for row in chorale:
    s.append(chord.Chord([n for n in row if n],quarterLength=1))
s.show('midi')

In [24]:
def generate_random_chorale(length, rest_probability=0.2, pitch_low=36, pitch_high=81, seed=None):
    rng = np.random.default_rng(seed)  # random number generator
    random_pitches = rng.integers(pitch_low, pitch_high + 1, size=(length, 4))  # generate random notes

    # some masking to have both silence and random pitches
    rest_mask = rng.random((length, 4)) < float(rest_probability)
    chorale = np.where(rest_mask, 0, random_pitches).astype(int)
    
    return chorale

In [25]:
chorale = generate_random_chorale(56).tolist()
s = stream.Stream()
for row in chorale:
    s.append(chord.Chord([n for n in row if n], quarterLength=1))
s.show('midi')

In [29]:

loss, accuracy = model.evaluate(X_test, y_test)

print(f"Loss: {loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")

34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.8172 - loss: 0.6672
Loss: 0.6672
Accuracy: 0.8172
